# Module `graph_generator.py`

Ce notebook explique pas a pas comment une instance CESIPATH est construite et **pourquoi chaque etape existe**.

**Probleme de fond** : on veut generer aleatoirement des instances VRP **realistes** (geographiquement plausibles, avec des contraintes routieres) et **resolubles** (pas d'isolement de clients, pas de capacite irrealisable). Or n'importe quel tirage aleatoire de graphe produit dans la majorite des cas un graphe **deconnecte**, qui n'admettrait aucune solution VRP. La litterature classique sur les graphes aleatoires (Erdos & Renyi [1]) montre que la transition connexe/non-connexe a lieu pour une densite tres precise (`p ~ ln(n)/n`), ce qui rend le tirage naif inutilisable. Il faut donc une procedure **structuree** qui garantit la connexite **par construction**, et pas par filtrage post-hoc.

Le generateur produit quatre objets consommes par le reste du projet :

1. le **graphe de base** : toutes les vraies routes physiques, sans contrainte ;
2. le **graphe residuel** : le meme graphe apres application des contraintes statiques (interdictions, surcouts) ;
3. le **graphe complete** : la matrice de couts obtenue par Dijkstra sur le residuel, utilisable directement par un solveur metrique ;
4. les **snapshots dynamiques** : etats du reseau a chaque tour de simulation, recalcules par `DynamicNetworkSimulator`.

**La cle de l'implementation** : le generateur garantit la **connexite** et les **bornes de densite** par **construction + rejet** (rejection sampling), jamais par post-traitement destructif. Cette approche evite les biais qu'introduiraient des "reparations" sur un graphe deconnecte (ajout d'aretes arbitraires qui fausseraient la distribution).

In [ ]:
import sys
from pathlib import Path
from pprint import pprint

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.dynamic_network import DynamicNetworkSimulator
from cesipath.graph_generator import GraphGenerator
from cesipath.metric_closure import check_triangle_inequality
from cesipath.models import EdgeStatus, GraphGenerationConfig

## 1. Parametres de generation

`GraphGenerationConfig` regroupe tous les leviers. Les valeurs par defaut suffisent la plupart du temps.

Parametres statiques cles :

- `node_count` : nombre de sommets (depot inclus).
- `seed` : graine de reproductibilite. Deux `generate()` avec la meme graine donnent la meme instance.
- `auto_density_profile=True` : choisit les bornes de densite selon `node_count` (graphes denses quand petits, creux quand grands).
- `forbidden_rate` : proba qu'une arete hors arbre couvrant soit statiquement interdite.
- `surcharge_rate` : proba qu'une arete recoive un surcout statique dans `[surcharge_min, surcharge_max]`.
- `vehicle_capacity`, `demand_min`, `demand_max` : donnent la contrainte de capacite du VRP.

Parametres dynamiques cles (utilises seulement par `DynamicNetworkSimulator`, pas par `generate()`) :

- `dynamic_sigma`, `dynamic_mean_reversion_strength`, `dynamic_max_multiplier` : loi d'evolution des couts ;
- `dynamic_forbid_probability`, `dynamic_restore_probability` : coupures/retablissements temporaires ;
- `dynamic_max_disabled_ratio` : part maximale d'aretes OFF tolerees a un instant donne.

In [ ]:
config = GraphGenerationConfig(
    node_count=7,
    seed=42,
    auto_density_profile=True,
    forbidden_rate=0.1,
    surcharge_rate=0.25,
    vehicle_capacity=40,
)

generator = GraphGenerator(config)
instance = generator.generate()

pprint(instance.summary())

## 2. Etape 1 - Coordonnees uniformes

`_generate_coordinates` tire chaque sommet uniformement dans le rectangle `[0, width] x [0, height]`. Les couts d'aretes utilisent la **distance euclidienne** (`math.dist`), arrondie a 2 decimales.

**Pourquoi euclidien ?** Trois raisons :

1. **L'inegalite triangulaire est triviale** entre points du plan : `d(A,C) <= d(A,B) + d(B,C)` est une propriete geometrique immediate. C'est un prerequis pour pouvoir verifier ensuite la propriete metrique du graphe complete (section 8).
2. **Reproduit les distances routieres reelles** au premier ordre : sur des reseaux de transport reels, les distances euclidiennes a vol d'oiseau sont fortement correlees aux distances reelles (etudes empiriques de Boscoe et al. [2] sur les detours moyens en milieu rural et urbain).
3. **Permet une visualisation directe** : les coordonnees suffisent a tracer le graphe sans projection supplementaire.

**Pourquoi pas un autre modele de positionnement** (ex. clusters, modeles spatiaux non uniformes) ? Pour rester dans le cadre du livrable 1 et garder la generation interpretable. Un generateur clusterise (clients groupes par "ville") serait plus realiste mais introduirait beaucoup de parametres et compliquerait l'analyse de complexite des algorithmes en fonction de `node_count`.

In [ ]:
print("Coordonnees :")
pprint(instance.coordinates)

## 3. Etape 2 - Arbre couvrant pour garantir la connexite

**Definition.** Un *arbre couvrant* (spanning tree) est un sous-ensemble d'aretes qui (a) connecte tous les sommets et (b) ne contient aucun cycle. Pour un graphe a `n` sommets, un arbre couvrant a exactement `n-1` aretes - c'est le **minimum** d'aretes necessaires pour connecter tous les sommets (Bondy & Murty [3, theoreme 2.1]).

**Construction.** Dans `_build_connected_adjacency`, on commence par attacher chaque nouveau sommet `i >= 1` a un parent aleatoire `< i`. Ce procede produit toujours un arbre couvrant : par recurrence, le sous-graphe des sommets `{0, ..., i}` est connexe a chaque etape, donc le graphe final l'est aussi.

**Pourquoi cette construction plutot que tirer des aretes au hasard et verifier ?** Parce qu'avec un tirage non structure, la probabilite d'obtenir un graphe **deconnecte** est elevee pour des densites faibles, et chaque rejet coute un cycle complet de generation. La construction par arbre **garantit** la connexite en O(n) etapes, sans aucun test ni rejet. C'est la premiere garantie structurelle du generateur : aucun tirage ulterieur (densification, contraintes statiques) ne peut detruire la connexite, parce qu'on ne **retire** jamais d'arete d'arbre - on ne fait qu'**ajouter** par-dessus.

In [ ]:
from cesipath.validators import is_connected

active = set(instance.base_edges.keys())
print("Nombre d'aretes base :", len(active))
print("Graphe de base connexe ?", is_connected(instance.node_count, active))

## 4. Etape 3 - Densification jusqu'a la cible

Une fois l'arbre place, on ajoute des aretes aleatoires jusqu'a atteindre `target_edges`, deduit de `resolved_edge_density` (ratio aretes presentes / aretes possibles).

**Pourquoi densifier ?** Un arbre couvrant donne un graphe **minimal** (n-1 aretes), ou il n'existe **qu'un seul chemin** entre toute paire de sommets. Pour un VRP, c'est insuffisant : le solveur n'aurait aucune flexibilite pour optimiser des tournees, et les contraintes dynamiques (interdictions temporaires) deconnecteraient immediatement le graphe. Densifier ajoute des **alternatives de chemins**, ce qui permet :

1. **A l'optimisation** : choisir entre plusieurs chemins de cout proche pour eviter les arcs surcharges ou maintenir un equilibre de charges entre tournees.
2. **A la robustesse dynamique** : meme si une arete devient interdite en cours de simulation, il reste un chemin de secours.

**Profil de densite auto** (`auto_density_profile=True`) : le generateur ajuste la densite cible **en fonction de `node_count`** - graphes denses quand petits (~70-80%), creux quand grands (~10-20%). Cette mise a l'echelle imite les reseaux reels : une petite ville rurale a peu de routes mais elles sont presque toutes interconnectees, tandis qu'un grand reseau routier a une densite globale faible (chaque ville n'a que quelques connexions directes a ses voisines geographiques). Cette propriete - reseau peu dense mais structure - est typique des reseaux de transport reels (Watts & Strogatz [4]).

In [ ]:
max_edges = instance.node_count * (instance.node_count - 1) // 2
print("Max d'aretes possibles :", max_edges)
print("Densite cible :", config.resolved_edge_density)
print("Densite base obtenue :", round(instance.base_density, 4))

## 5. Etape 4 - Arbre protege contre les contraintes statiques

**C'est le point le plus important du generateur.** Dans `_build_residual_edges`, on applique les tirages `FORBIDDEN` et `SURCHARGED` (Markov independants par arete avec probabilites `forbidden_rate` et `surcharge_rate`), **sauf sur les aretes de l'arbre couvrant**. Ces aretes restent toujours `FREE`.

**Pourquoi cette protection ?** Sans elle, les tirages de contraintes `FORBIDDEN` peuvent **detruire la connexite** du graphe residuel. Un cas pathologique simple : si une arete d'arbre est tiree FORBIDDEN, tout un sous-arbre devient inaccessible. L'instance generee serait invalide et tomberait dans la boucle de rejet (section 10), faisant grimper le nombre de tentatives jusqu'a `generation_max_attempts` voire un echec definitif des que `forbidden_rate` depasse quelques pourcents.

**Reformulation theorique.** Dans la theorie de la **percolation** sur graphes (Grimmett [5]), supprimer chaque arete avec probabilite `p` provoque la deconnexion globale des qu'on franchit le **seuil de percolation** `p_c`, qui est tres bas pour les graphes peu denses. Proteger l'arbre couvrant revient a forcer une "epine dorsale" indestructible qui maintient la connexite quel que soit `forbidden_rate`. C'est mathematiquement equivalent a faire de la percolation **uniquement sur les aretes hors-arbre**.

Consequence pratique : meme avec `forbidden_rate = 0.5`, le graphe residuel **reste connexe** par construction. Dijkstra trouvera donc toujours un chemin entre deux sommets apres filtrage des aretes interdites - propriete sans laquelle la fermeture metrique de la section 7 serait impossible.

In [ ]:
tree_edges = GraphGenerator._spanning_tree_edges(
    {key: edge.base_cost for key, edge in instance.base_edges.items()}
)
tree_with_status = [
    (key, instance.residual_edges[key].status.value) for key in tree_edges
]
print("Aretes de l'arbre couvrant et leur statut :")
pprint(tree_with_status)

non_free_in_tree = [k for k, status in tree_with_status if status != "libre"]
print("\nAretes de l'arbre NON libres (doit etre vide) :", non_free_in_tree)

## 6. Etape 5 - Trois matrices de couts

Le generateur construit **trois matrices distinctes**. Cette redondance n'est pas du gaspillage memoire : chaque matrice repond a une question differente.

- **`base_costs`** (graphe physique initial) : "quelles sont les vraies routes existantes, et combien coutent-elles si rien ne les contraignait ?". Cellule a 0 si pas d'arete directe. Sert a la **visualisation** du reseau de fond et au debug. Aucun solveur ne l'utilise directement.
- **`residual_costs`** (apres contraintes statiques) : "quelles sont les routes praticables maintenant, et a quel cout ?". Aretes FORBIDDEN retirees (valeur 0), aretes SURCHARGED multipliees. C'est la **vraie topologie courante** du reseau. Sert au calcul de la fermeture metrique.
- **`completed_costs`** (matrice complete par Dijkstra) : "quel est le cout du **meilleur trajet possible** entre i et j, en composant des aretes residuelles si necessaire ?". Cellule `(i, j)` = cout du plus court chemin. C'est ce qu'un solveur metrique attend en entree.

**Pourquoi separer plutot que tout deriver a la volee ?** Parce que les solveurs metaheuristiques font des **millions** d'evaluations de cout d'arete (chaque mouvement local, chaque calcul de delta). Recalculer un Dijkstra a chaque acces serait inutilisable en performance. La separation reflete le pattern **separation precalcul / hot loop** standard en optimisation combinatoire (Cordeau et al. [6, section 2]).

Les solveurs consomment uniquement `completed_costs` (via `SolverInput`, voir `solver_input.ipynb`). Les autres matrices restent accessibles pour le debug, la visualisation et la validation.

In [ ]:
def show(title, matrix):
    print(title)
    for row in matrix:
        print(row)
    print()

show("base_costs :", instance.base_costs)
show("residual_costs :", instance.residual_costs)
show("completed_costs :", instance.completed_costs)

## 7. Etape 6 - Fermeture metrique (Dijkstra)

`complete_graph_with_shortest_paths` (dans `metric_closure.py`) applique l'**algorithme de Dijkstra** [7] depuis chaque source et produit deux choses :

- la matrice `completed_costs` ;
- un dictionnaire `completed_paths[(i, j)]` qui contient le **vrai chemin** passant par les vraies aretes residuelles.

**Pourquoi Dijkstra et pas Floyd-Warshall ?** Floyd-Warshall calcule toutes les distances en une passe O(n^3). Dijkstra de chaque source coute O(n * (m + n log n)) avec un tas binaire (Cormen et al. [8, chap. 24]). Pour les graphes peu denses (m << n^2) qu'on genere ici, Dijkstra est plus rapide. Surtout, Dijkstra produit naturellement les **predecesseurs** qui permettent de reconstruire les chemins, alors que Floyd-Warshall demanderait une matrice supplementaire de reconstruction.

**Pourquoi conserver les chemins reels ?** Parce qu'un solveur metrique raisonne sur la **matrice completee** : il peut proposer la sequence virtuelle `0 -> 3 -> 5`, mais le vrai deplacement passe peut-etre par `0 -> 4 -> 3 -> 7 -> 5`. Sans les chemins reels, on ne pourrait :

- ni **visualiser** la solution sur le vrai reseau routier (les fleches sauteraient des sommets),
- ni **executer la solution dynamique** (`execute_dynamic`) qui doit savoir quelles aretes reelles seront empruntees pour detecter d'eventuels conflits avec les arcs FORBIDDEN qui apparaissent en cours de simulation.

In [ ]:
for target in range(1, instance.node_count):
    path = instance.completed_paths.get((0, target))
    cost = instance.completed_costs[0][target]
    print(f"0 -> {target} : cout={cost} via {path}")

## 8. Inegalite triangulaire

**Definition.** L'inegalite triangulaire dit que pour tout triplet de sommets `(i, j, k)` :

```text
d(i, j) <= d(i, k) + d(k, j)
```

Autrement dit, le chemin direct est toujours au moins aussi court qu'un detour.

**Pourquoi cette propriete est-elle automatique ici ?** Parce que `completed_costs[i][j]` est defini comme **le plus court chemin** entre i et j. S'il existait un detour `i -> k -> j` de cout strictement inferieur, ce serait alors **lui** le plus court chemin, contradiction. C'est une propriete **structurelle** de la fermeture metrique - on ne fait pas confiance, on verifie quand meme via `check_triangle_inequality` parce qu'un bug d'implementation de Dijkstra (mauvaise gestion d'aretes negatives, oubli d'un noeud) la violerait silencieusement.

**Pourquoi c'est important pour les algorithmes ?** Beaucoup d'heuristiques pour le TSP/VRP **supposent** l'inegalite triangulaire et perdent leurs garanties sans elle. Exemples :

- **2-opt** : sans inegalite triangulaire, l'echange d'aretes peut produire un cout arbitrairement plus eleve, l'operateur n'est plus une descente fiable.
- **Christofides** [9] : son ratio d'approximation 3/2 ne tient que sur un TSP metrique.
- **Split de Prins** [10] : la propriete d'optimalite du decoupage suppose des couts metriques.

C'est donc un garde-fou essentiel : `check_triangle_inequality` doit toujours renvoyer `True` sur une instance correctement generee.

In [ ]:
ok, violation = check_triangle_inequality(instance.completed_costs)
print("Inegalite triangulaire respectee ?", ok)
print("Violation eventuelle :", violation)

## 9. Demandes uniformes

`_generate_demands` tire **une seule** valeur entiere dans `[demand_min, demand_max]` et l'affecte a tous les clients. Le depot garde `0`.

**Pourquoi uniforme et pas une demande aleatoire par client ?** Le livrable 1 fixe cette hypothese pour deux raisons :

1. **Simplicite analytique** : avec une demande uniforme, le **nombre minimal de tournees** se calcule directement par `ceil(n_clients * demande / capacite)`. Avec des demandes heterogenes, ce calcul devient un **bin-packing** NP-difficile (Toth & Vigo [11, chap. 1]) - il faudrait resoudre un sous-probleme pour avoir cette borne.
2. **Comparaison equitable des algorithmes** : sur des demandes heterogenes, les performances des metaheuristiques varient enormement selon la distribution. Sur des demandes uniformes, on isole la difficulte **purement combinatoire** du routage (ordre de visite, decoupage en tournees) sans confondre avec la difficulte du packing.

Le VRP avec demandes uniformes reste **NP-difficile** (c'est une generalisation directe du TSP), donc on ne perd pas la difficulte interessante. C'est juste qu'on l'isole proprement.

In [ ]:
print("Demande uniforme :", instance.uniform_demand)
print("Capacite vehicule :", config.vehicle_capacity)
print("Nombre minimal de tournees :", instance.minimum_route_count)

## 10. Boucle de rejet (rejection sampling)

**Principe.** Le **rejection sampling** est une technique standard pour echantillonner d'une distribution conditionnelle quand on ne sait pas la simuler directement (Robert & Casella [12, chap. 2]). Ici : on ne sait pas tirer directement un graphe satisfaisant *toutes* les contraintes de densite et de degre, donc on **tire dans une distribution simple** (le procede arbre + densification + contraintes statiques) puis on **rejette** les tirages qui ne respectent pas les criteres.

`generate()` construit une instance candidate, puis la passe a `InstanceValidator.is_valid`. Quatre criteres doivent etre respectes :

1. densite de base dans `[resolved_min_base_density, resolved_max_base_density]` ;
2. densite residuelle dans `[resolved_min_residual_density, resolved_max_residual_density]` ;
3. degre moyen residuel `>= resolved_min_average_residual_degree` (chaque sommet doit avoir suffisamment de voisins pour offrir des alternatives de chemins) ;
4. graphe residuel **connexe** (apres retrait des FORBIDDEN).

Si un critere echoue, on recommence avec une nouvelle graine interne, jusqu'a `generation_max_attempts` tentatives. **Si rien ne passe**, une `ValueError` est levee : c'est le signe que les bornes demandees sont **infaisables** pour cette taille - aucun graphe n'existe en respectant simultanement toutes les contraintes. La cellule suivante force un echec volontaire en imposant une densite residuelle de 95-100% sur 5 sommets : impossible des qu'on tire ne serait-ce qu'une seule arete FORBIDDEN.

**Pourquoi pas modifier le generateur pour reparer les rejets plutot que rejeter ?** Parce que la reparation introduit un biais difficile a controler : ajouter une arete pour atteindre la densite minimale fausse la distribution finale. Le rejection sampling preserve la propriete : **conditionnellement aux contraintes**, la distribution reste celle du procede de base (Robert & Casella [12, theoreme 2.1]).

In [ ]:
from cesipath.models import GraphGenerationConfig

impossible = GraphGenerationConfig(
    node_count=5,
    auto_density_profile=False,
    min_residual_density=0.95,
    max_residual_density=1.0,
    generation_max_attempts=5,
    seed=0,
)

try:
    GraphGenerator(impossible).generate()
except ValueError as err:
    print("Rejet attendu :", err)

## 11. Passage a la dynamique

Une fois l'instance statique validee, la simulation dynamique prend le relais via `DynamicNetworkSimulator`.

**Pourquoi dynamique ?** Le VRP-CDR n'est pas un probleme statique : il modelise la realite ou les temps de trajet varient (trafic, meteo, travaux) et ou des routes peuvent devenir indisponibles temporairement (accidents, pannes, restrictions horaires). Un solveur qui optimise une seule fois a `t=0` produirait des tournees sous-optimales des que les conditions evoluent.

Un `DynamicGraphSnapshot` represente l'**etat du reseau a un tour t** (non immuable, reconstruit a chaque pas). Il stocke :

- `edge_costs` : cout courant de chaque arete (apres evolution gaussienne avec retour a la moyenne, voir `dynamic_costs.ipynb`) ;
- `edge_availability` : booleen par arete indiquant si elle est ON ou OFF temporairement (le statut dynamique s'empile sur le statut statique, voir `dynamic_network.ipynb`) ;
- `residual_costs`, `completed_costs`, `completed_paths` : recalcules apres chaque tour pour que les solveurs disposent d'une matrice metrique coherente avec l'etat courant.

`advance()` applique **tirage + validation + recalcul**. Les details sont dans `dynamic_network.ipynb`. La cellule suivante montre un passage `t=0 -> t=1` avec observation du changement de la premiere ligne de la matrice completee.

In [ ]:
simulator = DynamicNetworkSimulator(instance, seed=42)
snapshot = simulator.initialize_snapshot()
print("Tour initial :", snapshot.step)
print("Aretes actives :", snapshot.active_edge_count)

next_snapshot = simulator.advance(snapshot)
print("Tour suivant :", next_snapshot.step)
print("Aretes actives :", next_snapshot.active_edge_count)
print("\nNouvelle matrice complete ligne 0 :", next_snapshot.completed_costs[0])

## References

[1] **Erdos, P. & Renyi, A. (1960).** *On the evolution of random graphs.* Publications of the Mathematical Institute of the Hungarian Academy of Sciences, 5(1), 17-60.

[2] **Boscoe, F. P., Henry, K. A. & Zdeb, M. S. (2012).** *A nationwide comparison of driving distance versus straight-line distance to hospitals in the United States.* The Professional Geographer, 64(2), 188-196. https://doi.org/10.1080/00330124.2011.583586

[3] **Bondy, J. A. & Murty, U. S. R. (2008).** *Graph Theory.* Springer (Graduate Texts in Mathematics, vol. 244). https://doi.org/10.1007/978-1-84628-970-5

[4] **Watts, D. J. & Strogatz, S. H. (1998).** *Collective dynamics of 'small-world' networks.* Nature, 393(6684), 440-442. https://doi.org/10.1038/30918

[5] **Grimmett, G. (1999).** *Percolation* (2nd ed.). Springer. https://doi.org/10.1007/978-3-662-03981-6

[6] **Cordeau, J.-F., Gendreau, M., Laporte, G., Potvin, J.-Y. & Semet, F. (2002).** *A guide to vehicle routing heuristics.* Journal of the Operational Research Society, 53(5), 512-522. https://doi.org/10.1057/palgrave.jors.2601319

[7] **Dijkstra, E. W. (1959).** *A note on two problems in connexion with graphs.* Numerische Mathematik, 1(1), 269-271. https://doi.org/10.1007/BF01386390

[8] **Cormen, T. H., Leiserson, C. E., Rivest, R. L. & Stein, C. (2009).** *Introduction to Algorithms* (3rd ed.). MIT Press.

[9] **Christofides, N. (1976).** *Worst-case analysis of a new heuristic for the travelling salesman problem.* Technical Report 388, Carnegie Mellon University.

[10] **Prins, C. (2004).** *A simple and effective evolutionary algorithm for the vehicle routing problem.* Computers & Operations Research, 31(12), 1985-2002. https://doi.org/10.1016/S0305-0548(03)00158-8

[11] **Toth, P. & Vigo, D. (eds.) (2014).** *Vehicle Routing: Problems, Methods, and Applications* (2nd ed.). SIAM. https://doi.org/10.1137/1.9781611973594

[12] **Robert, C. P. & Casella, G. (2004).** *Monte Carlo Statistical Methods* (2nd ed.). Springer. https://doi.org/10.1007/978-1-4757-4145-2
